# Photometry

#### There is more than one way to measure photometry 

#### Large circular aperture has been our default so far (i.e. sum counts in a large radius around the object)

#### Small circular aperture is a common option for clusters (i.e. sum counts in a small radius around the object)

#### Elliptical aperture is an option if your object is not circular (i.e. galaxies)

#### Model photometry is an option if you can model the shape of your object (discussed as part of catalogues and source detection)

####

# Large aperture photometry 

#### A very large aperture can capture the vast majority of the light from each object 

#### Good for avoiding systematic errors 

#### For faint objects you can include a lot of noisy pixels

#### For some faint objects the noise can sawmp the signal - you can even get negative fluxes 

#### A large aperture can also catch light from neighbouring objects 

####

# Small aperture photometry 

#### A small aperture can have a relatively high signal-to-noise as you're only measuring the brightest pixels for an object

#### Good for faint objects where large aperture photometry get swamped by noise 

#### Lowers the risk of errors due to light from neighbouring objects 

#### A risk is systematic errors as you may only be capturing a fraction of the light

#### Caution required mixing photometry from difference apertures - if there's an offset you need to account for it 

#### For example, if you calibrate with large apertures and measure with small aperture you may get systematically faint magnitudes

#### If you use the same aperture for your calibration and science then you should be OK

#### Is sensitive to variations in image quality across the field 

####


# Elliptical aperture photometry 

#### If your object is not circularly symmetric, you can use another aperture shape

#### Elliptical aperture are often used for galaxies (but rectangular apertures can be used too)

#### An aperture can be defined manually

#### An aperture can be defined using the light distribution of the object (2nd order image moments) 

####

# Isophotal photomery 

#### Add up the light from pixels significantly above the background 

#### Doesn't make assumptions about the object shape per se

#### Great for bright objects with high surface brightness 

#### Systematically underestimates the flux of faint objects where only a small fraction of the light is in high signal-to-noise pixels

#### Systematic errors depend on the image quality and depth

#### 

# Lets load an image and generate a catalogue 

#### An advantage of python vs interactive codes is you can quickly rerun it

In [ ]:
# Import various libraries 

import numpy as np
import astropy
import photutils
import ccdproc
from ccdproc import CCDData, combiner
from astropy import units as u
import astropy.io.fits as fits
from astropy.io import ascii
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
from photutils.centroids import centroid_com, centroid_1dg, centroid_2dg
from photutils.aperture import CircularAperture, EllipticalAperture
from photutils.aperture import aperture_photometry
from photutils.segmentation  import detect_sources, deblend_sources, SourceCatalog
import gc                               

from astropy.coordinates import SkyCoord
from astroquery.gaia import Gaia


In [ ]:
fn='NGC_3293_V_median_wcs.fits'
science_im=CCDData.read(fn, unit = "adu")                           

w = science_im.wcs
print(w)

skydata=science_im.data
print(skydata[0])
skydata=science_im.data[1030:1130,60:160]
p1=np.nanpercentile(skydata, 1)
p99=np.nanpercentile(skydata, 99)
plt.imshow(skydata, cmap='gray', vmin=p1, vmax=p99)
ax = plt.gca()
ax.invert_yaxis()
plt.colorbar(shrink=0.8)



In [ ]:
# Lets run stats on some of this data to get the background and noise

med=np.nanmedian(skydata)
print('Median:', med)

std=np.nanstd(skydata)
print('Standard deviation:', std)


In [ ]:
segimage=detect_sources(science_im.data, 2.00*std, 9, connectivity=4, mask=None)
print('Segmentation image minimum:', np.min(segimage.data))
print('Segmentation image maximum:', np.max(segimage.data))


In [ ]:
source_table=SourceCatalog(science_im.data, segimage, error=None, mask=None, background=None, wcs=w)
print('Source table length:\n', len(source_table))
print('\n\n')
print('First entry centroid x and y values :') 
print(source_table[0].centroid[0], source_table[0].centroid[1])
print('\n\n')
print('First source sky icrs coordinates :') # 
print(source_table[0].sky_centroid_icrs)
print(source_table[0].sky_centroid_icrs.ra, source_table[0].sky_centroid_icrs.dec)


In [ ]:
positions = []   

for star in source_table:
    positions.append((star.centroid[0], star.centroid[1]))
print(positions[0:10]) # delete 


In [ ]:

big_apertures = CircularAperture(positions, r=15.0)
big_phot_table = aperture_photometry(science_im-np.nanmedian(science_im.data), big_apertures)
print(big_phot_table[200:210])


In [ ]:
small_apertures = CircularAperture(positions, r=6.0)
small_phot_table = aperture_photometry(science_im-np.nanmedian(science_im.data), small_apertures)
print(small_phot_table[200:210])


# Lets compare big and small aperture photometry 


In [ ]:
# Lets compare big and small aperture photometry 

px=[]
py=[]

for idx, phot in enumerate(small_phot_table):
    if phot['aperture_sum']>0.0 and phot['aperture_sum']<1.0e+7 :
        px.append(np.log10(phot['aperture_sum']))
        py.append(small_phot_table[idx]['aperture_sum']/big_phot_table[idx]['aperture_sum'])

plt.scatter(px,py)
ax = plt.gca()
ax.set_xlim([3, 7])
ax.set_ylim([0.5, 1.5])        
plt.xlabel('Small aperture log flux')
plt.ylabel('Small aperture flux divided by large aperture flux')


# Isophotal photometry

In [ ]:
print('Image axes\n') 
ximax=len(science_im.data[0])
yimax=len(science_im.data)

print('Source table length:\n', len(source_table))
isoflux=[0.0]*len(source_table)
print(isoflux[0:10])
imdata=science_im-np.nanmedian(science_im.data)

# Step through the segmentation image identifying the pixels associated with objects 
yi=0
while yi<yimax: 
    xi=0
    while xi<ximax:
        idx=segimage.data[yi,xi]-1
        if idx>=0:
            # print(xi,yi)
            # print(imdata[yi,xi])
            isoflux[idx]=isoflux[idx]+imdata[yi,xi]
        xi=xi+1
    yi=yi+1

print(isoflux[200:210])


# Lets compare isophotal and small aperture photometry 


In [ ]:
# Lets compare big and small aperture photometry 

px=[]
py=[]

for idx, phot in enumerate(small_phot_table):
    if phot['aperture_sum']>0.0 and phot['aperture_sum']<1.0e+7 :
        px.append(np.log10(phot['aperture_sum']))
        py.append(isoflux[idx]/phot['aperture_sum'])

plt.scatter(px,py)
ax = plt.gca()
ax.set_xlim([3, 7])
ax.set_ylim([0.5, 1.5])        
plt.xlabel('Small aperture log flux')
plt.ylabel('Iso flux divided by small aperture flux')


# Example of large elliptical aperture photometry 


In [ ]:
# Example of large elliptical aperture photometry 

ellpositions = []   

ellpositions.append((845, 615)) # Position - can have more positons 
a=600                  # Major axis
b=400                  # Minor axis
theta=120*np.pi/180.0  # Measured couterclockwise from x-axis
ell_apertures = EllipticalAperture(ellpositions, a, b, theta=theta)

ell_phot_table = aperture_photometry(science_im-np.nanmedian(science_im.data), ell_apertures)

print(ell_phot_table)


In [ ]:
# In a previous video we found the zeropoint of this image to be 25.916296772693492

zp=25.916296772693492

mag = zp - 2.5*np.log10(ell_phot_table[0]['aperture_sum'])
print('V-mag for much of the cluster:', mag)

# How does this compare with the literature 